# Fase 1 — Máscaras de referência (GEE)

Construção das máscaras de referência candidatas para a classe **café** na área de estudo (Região Geográfica Imediata de Guaxupé - MG).

Este estágio prepara duas fontes: a classificação **MapBiomas** (classe café = 46) binarizada a 10 m e um agrupamento não supervisionado (**k-means**) sobre os embeddings anuais **AlphaEarth**. Ambas são exportadas como GeoTIFF para o Google Drive e servirão de base à comparação de fontes de verdade de campo na Fase 2.

## Obtenção do repositório

Garante a presença do pacote `src/` numa área gravável da plataforma: no Colab clona o repositório público para `/content` e atualiza em execuções seguintes via `git pull`; no Kaggle copia o dataset somente-leitura de `/kaggle/input` para `/kaggle/working` (ou clona do GitHub se não houver dataset). Se a raiz já estiver acessível, a etapa é ignorada. No ambiente local não é necessária, pois o repositório já está no diretório corrente.

In [ ]:
import importlib.util
import os
import shutil
import subprocess
from pathlib import Path


# URL pública do repositório, usada como fallback quando não há dataset montado.
REPO_URL = "https://github.com/jotap1101/tcc.git"
REPO_NAME = "tcc"


def _copy_repo(source: Path, dest: Path) -> None:
    """Copia o repositório para a área gravável, ignorando metadados de versionamento."""
    dest.mkdir(parents=True, exist_ok=True)
    for item in source.iterdir():
        if item.name in {".git", "__pycache__", ".ruff_cache", ".mypy_cache", ".pytest_cache"}:
            continue
        target = dest / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)


# Detecta Colab com segurança, mesmo quando o pacote google não existe.
is_colab = "COLAB_GPU" in os.environ
if not is_colab:
    try:
        is_colab = importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        is_colab = False

# Resolve a área gravável e a fonte do repositório conforme a plataforma.
if is_colab:
    working_dir = Path("/content") / REPO_NAME
    repo_source = None
elif Path("/kaggle").is_dir():
    working_dir = Path("/kaggle/working") / REPO_NAME
    repo_source = next(
        (p for p in Path("/kaggle/input").glob("*") if (p / "src" / "config.yaml").is_file()),
        None,
    )
else:
    working_dir = None
    repo_source = None

# Sincroniza o repositório: ignora se já presente, atualiza via git pull,
# copia o dataset montado ou clona do GitHub.
if working_dir is not None:
    if (working_dir / "src" / "config.yaml").is_file():
        print(f"Repositório já presente em {working_dir}.")
    elif (working_dir / ".git").is_dir():
        subprocess.run(["git", "-C", str(working_dir), "pull", "--ff-only"], check=True)
        print(f"Repositório atualizado em {working_dir}.")
    elif repo_source is not None:
        _copy_repo(repo_source, working_dir)
        print(f"Repositório copiado de {repo_source} para {working_dir}.")
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(working_dir)], check=True)
        print(f"Repositório clonado em {working_dir}.")
    os.environ["TCC_ROOT"] = str(working_dir)
else:
    print("Ambiente local: repositório já disponível no diretório corrente.")


## Detecção da raiz do repositório

Localiza a raiz do repositório pelo marcador `src/config.yaml` nos diretórios corrente, ancestrais e raízes de montagem das plataformas de nuvem (Kaggle/Colab), além do override via variável de ambiente `TCC_ROOT`. A raiz é inserida no caminho de importação, garantindo o acesso ao pacote `src/`.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


# Raízes de busca: env TCC_ROOT, diretório corrente com ancestrais e montagens
# de repositórios nas plataformas de nuvem (Kaggle /kaggle/input, Colab /content).
def _candidate_bases() -> list[Path]:
    bases: list[Path] = []
    tcc_root = os.environ.get("TCC_ROOT")
    if tcc_root:
        bases.append(Path(tcc_root))
    bases.extend([Path.cwd(), *Path.cwd().parents])
    for mount in (Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")):
        if mount.is_dir():
            bases.append(mount)
    return bases


# Verifica a base e seus subdiretórios imediatos em busca do marcador da raiz.
def _find_project_root() -> Path:
    for base in _candidate_bases():
        for candidate in [base, *base.glob("*")]:
            if candidate.is_dir() and (candidate / "src" / "config.yaml").is_file():
                return candidate
    raise RuntimeError("Raiz do repositório não localizada (src/config.yaml ausente).")


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Raiz do projeto: {PROJECT_ROOT}")


## Detecção da plataforma

Identifica o ambiente de execução (Kaggle, Colab ou local) para adaptar a instalação de dependências e a leitura de segredos.

In [ ]:
import importlib.util
import os


def _is_colab_runtime() -> bool:
    """Detecta Colab com segurança, mesmo quando o pacote google não existe."""
    if "COLAB_GPU" in os.environ:
        return True
    try:
        return importlib.util.find_spec("google.colab") is not None
    except ModuleNotFoundError:
        return False


# Colab é detectado primeiro, pois /kaggle também existe nos runtimes do Colab.
def detect_platform() -> str:
    if _is_colab_runtime():
        return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle/working").is_dir():
        return "kaggle"
    return "local"


PLATFORM = detect_platform()
print(f"Plataforma detectada: {PLATFORM}")

## Instalação condicional das dependências

Em Kaggle/Colab instala o pacote com os extras geoespaciais e de aprendizado de máquina. No ambiente local a instalação é ignorada, pois é gerenciada por `uv` e pelo CI.

In [ ]:
import subprocess

# Instala o projeto editavelmente com os extras necessários apenas em nuvem.
if PLATFORM in {"kaggle", "colab"}:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[geo,ml]"],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print("Dependências instaladas.")
else:
    print("Ambiente local: instalação ignorada (gerenciada por uv/CI).")

## Carregamento da configuração única

Lê a configuração de `src/config.yaml` por meio de `src/config.py`, fonte única de verdade dos assets de máscara, do ano de referência e dos parâmetros do k-means.

In [ ]:
from src.config import CONFIG

# Exibe os parâmetros das máscaras de referência.
print(f"Ano de referência: {CONFIG.get('masks.year')} | Classe café: {CONFIG.get('masks.coffee_class')}")
print(f"MapBiomas 10 m: {CONFIG.get('masks.mapbiomas_10m_asset')}")
print(f"AlphaEarth: {CONFIG.get('masks.alphaearth_collection')} | clusters: {CONFIG.get('masks.alphaearth_clusters')}")

## Fixação das sementes

Fixa as sementes de `python`, `numpy`, `torch` e `cuda`; a semente também alimenta a amostragem do k-means, tornando a máscara candidata reprodutível.

In [ ]:
from src.config import seed_everything

# Aplica a semente global definida na configuração.
resolved_seed = seed_everything()
print(f"Sementes fixadas em {resolved_seed}.")

## Carregamento de segredos

Em Kaggle/Colab os segredos são cadastrados no cofre da plataforma (Colab: painel Segredos na barra lateral; Kaggle: Add-ons → Secrets) e injetados nas variáveis de ambiente esperadas pelo pacote. A autenticação do Earth Engine usa exclusivamente OAuth de usuário: `GEE_OAUTH_CREDENTIALS_JSON`, além de `GEE_PROJECT` (obrigatório). Nenhum valor é impresso. No ambiente local, os segredos devem vir de variáveis de ambiente ou do arquivo `.env`.

In [ ]:
# Nomes das variáveis de ambiente consumidas pela aquisição.
GEE_AUTH_SECRETS = ("GEE_PROJECT", "GEE_OAUTH_CREDENTIALS_JSON")


def _load_secret(name: str) -> str:
    # Lê do cofre da plataforma (Colab ou Kaggle) e retorna o valor do segredo.
    if PLATFORM == "kaggle":
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    from google.colab import userdata

    return userdata.get(name)


if PLATFORM in {"kaggle", "colab"}:
    loaded: set[str] = set()
    for name in GEE_AUTH_SECRETS:
        try:
            os.environ[name] = _load_secret(name)
            loaded.add(name)
        except Exception:
            pass

    missing = [name for name in GEE_AUTH_SECRETS if name not in loaded]
    if missing:
        print("Segredos obrigatórios ausentes:", ", ".join(missing))
        print("Sem eles, a autenticação no Earth Engine falhará nas próximas células.")
else:
    print("Ambiente local: segredos esperados via variáveis de ambiente/.env.")

## Autenticação no Google Earth Engine

Inicializa o Earth Engine com credenciais OAuth de usuário lidas do cofre da plataforma (GEE_OAUTH_CREDENTIALS_JSON) ou do arquivo local gerado pelo `ee.Authenticate()`. No Colab, se nenhuma credencial existir, o fluxo de autorização é disparado automaticamente na célula. Sem credenciais, a fase não prossegue e um aviso é impresso.

In [ ]:
from src.data.gee_client import GEECredentialsError, init_ee


def _ensure_gee_credentials() -> None:
    # No Colab, dispara o fluxo OAuth oficial se as credenciais não existirem.
    if PLATFORM != "colab":
        return
    creds_path = Path.home() / ".config" / "earthengine" / "credentials"
    has_oauth = (
        os.environ.get("GEE_OAUTH_CREDENTIALS_JSON")
        or os.environ.get("GEE_OAUTH_CREDENTIALS_PATH")
        or creds_path.is_file()
    )
    if has_oauth:
        if creds_path.is_file():
            os.environ["GEE_OAUTH_CREDENTIALS_PATH"] = str(creds_path)
        return
    import ee

    ee.Authenticate()  # abre a janela de autorização do Google no Colab
    if creds_path.is_file():
        os.environ["GEE_OAUTH_CREDENTIALS_PATH"] = str(creds_path)
        print("Credenciais OAuth geradas automaticamente em:", creds_path)


_ensure_gee_credentials()
try:
    ee = init_ee()
    print("Earth Engine autenticado com sucesso.")
except GEECredentialsError as exc:
    print(f"Earth Engine NÃO autenticado: {exc}")


## Carregamento da área de estudo

Carrega a malha vetorial do IBGE, recorta a Região Geográfica Imediata de Guaxupé e converte a geometria para o formato do Earth Engine.

In [ ]:
from src.data.aoi import geometry_to_ee, get_region_geometry

# Obtém a geometria unificada da região e a converte para ee.Geometry.
aoi_geometry = get_region_geometry()
aoi_ee = geometry_to_ee(ee, aoi_geometry)
print(f"Área de estudo: {CONFIG.get('aoi.region_name')} ({CONFIG.get('aoi.region_code')})")

## Construção da máscara MapBiomas

Seleciona a classificação MapBiomas do ano de referência e a binariza, atribuindo 1 aos pixels da classe café (46) e 0 aos demais.

In [ ]:
from src.data.mask_utils import build_mapbiomas_coffee_mask

# Gera a máscara binária de café a partir da classificação MapBiomas.
mapbiomas_mask = build_mapbiomas_coffee_mask(ee, aoi_ee)
print(f"Banda da máscara MapBiomas: {mapbiomas_mask.bandNames().getInfo()}")

## Verificação da distribuição de classes

Calcula o histograma de frequência da máscara MapBiomas na área de estudo para confirmar a presença de pixels de café antes da exportação.

In [ ]:
# Conta pixels por valor (0 e 1) dentro da área de estudo.
histogram = mapbiomas_mask.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi_ee,
    scale=CONFIG.get("gee.scale_m"),
    maxPixels=CONFIG.get("gee.max_pixels"),
    bestEffort=True,
).getInfo()
print(f"Histograma da máscara MapBiomas: {histogram}")

## Mosaico dos embeddings AlphaEarth

Monta o mosaico anual dos embeddings AlphaEarth (64 bandas) recortado pela área de estudo, usado como representação não supervisionada do uso do solo.

In [ ]:
from src.data.mask_utils import get_alphaearth_embedding

# Mosaico anual dos embeddings AlphaEarth para o ano de referência.
embedding = get_alphaearth_embedding(ee, aoi_ee)
print(f"Número de bandas do embedding: {embedding.bandNames().size().getInfo()}")

## Clusterização k-means dos embeddings

Treina um clusterizador k-means sobre amostras dos embeddings e atribui um rótulo de cluster a cada pixel, gerando a máscara candidata não supervisionada.

In [ ]:
from src.data.mask_utils import cluster_alphaearth

# Agrupa os embeddings com semente fixa para reprodutibilidade.
clusters = cluster_alphaearth(ee, embedding, aoi_ee)
print(f"Banda de clusters: {clusters.bandNames().getInfo()}")

## Exportação das máscaras candidatas

Dispara as exportações GeoTIFF da máscara MapBiomas e do mapa de clusters AlphaEarth para o Google Drive, com descrições determinísticas.

In [ ]:
from src.data.gee_client import export_image_to_drive, make_export_description

# Exporta a máscara MapBiomas binária de café.
mapbiomas_description = make_export_description(
    f"{CONFIG.get('gee.export_prefix')}_mapbiomas_coffee",
    CONFIG.get("aoi.region_code"),
    CONFIG.get("gee.start_date"),
    CONFIG.get("gee.end_date"),
)
mapbiomas_task = export_image_to_drive(
    ee,
    mapbiomas_mask,
    description=mapbiomas_description,
    region=aoi_ee,
    subfolder=CONFIG.get("masks.export_subfolder"),
    file_name_prefix=mapbiomas_description,
)
print(f"Tarefa MapBiomas: {mapbiomas_description} | id={mapbiomas_task.id}")

## Exportação do mapa de clusters AlphaEarth

Dispara a exportação GeoTIFF do mapa de clusters AlphaEarth, que será comparado às demais fontes na Fase 2.

In [ ]:
# Exporta o mapa de clusters AlphaEarth como máscara candidata.
alphaearth_description = make_export_description(
    f"{CONFIG.get('gee.export_prefix')}_alphaearth_clusters",
    CONFIG.get("aoi.region_code"),
    CONFIG.get("gee.start_date"),
    CONFIG.get("gee.end_date"),
)
alphaearth_task = export_image_to_drive(
    ee,
    clusters,
    description=alphaearth_description,
    region=aoi_ee,
    subfolder=CONFIG.get("masks.export_subfolder"),
    file_name_prefix=alphaearth_description,
)
print(f"Tarefa AlphaEarth: {alphaearth_description} | id={alphaearth_task.id}")

## Acompanhamento das tarefas de exportação

Lista o estado das tarefas do Earth Engine para confirmar a conclusão das exportações das máscaras candidatas.

In [ ]:
import time

# Consulta o estado das tarefas recentes até todas concluírem.
while True:
    statuses = [t.status() for t in ee.batch.Task.list()[:5]]
    states = [s.get("state") for s in statuses]
    print(f"Estados: {states}")
    if all(state in {"COMPLETED", "FAILED", "CANCELLED"} for state in states):
        break
    time.sleep(30)